# Lineare Regression mit Statsmodels


In den ersten Notebooks dieses Repositorys haben wir mit der Bibliothek `scikit-learn` lineare Regressionsmodelle trainiert. In diesem Notebook stellen wir dir eine weitere Bibliothek vor, die du dafür nutzen kannst: `statsmodels`.

Welche Bibliothek du wählst, hängt eng mit deinem Ziel zusammen. Hier sind einige Vorteile und Einsatzbereiche der beiden Bibliotheken:

**scikit-learn:**

* bietet eine große Auswahl an Modellen und Algorithmen, nicht nur für Regression, sondern auch für Klassifikation, Dimensionsreduktion, Clustering usw.
* klare und leicht erlernbare Syntax
* Schwerpunkt auf prädiktiver Modellierung (predictive modeling) statt auf deskriptiver Statistik
* bietet Werkzeuge für Transformations-Pipelines


**statsmodels:**

* vielseitiges Modul für komplexe Statistik
* bietet leistungsstarke Werkzeuge für Statistik und Ökonometrie
* gut geeignet, um statistische Modelle zu fitten, statistische Tests durchzuführen oder Daten zu explorieren
* Schwerpunkt darauf, die relevanten Variablen und ihre Effektstärke (effect size) zu verstehen
* nützlich für Zeitreihen (time series)



Wie du siehst, haben beide Bibliotheken ihre Vorteile. Kurz gesagt: Wenn du deine Daten analysieren willst (z. B. die statistische Signifikanz verschiedener Variablen für dein Modell prüfen), ist statsmodels die richtige Wahl. Wenn dein Fokus darauf liegt, ein gutes Modell zu fitten und Vorhersagen zu treffen, ist scikit-learn die bessere Wahl.


## Lernziele

Am Ende dieses Notebooks solltest du in der Lage sein:

* ein lineares Regressionsmodell mit statsmodels zu trainieren.
* die Ergebnisse eines linearen Regressionsmodells mit statsmodels zu interpretieren.

## Der Cars-Datensatz



Wir bleiben beim inzwischen hoffentlich bekannten Cars-Datensatz und trainieren damit eine einfache und eine multiple lineare Regression mit statsmodels.

In [ ]:
# Pakete importieren
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [ ]:
# Wir nutzen denselben Auto-Datensatz wie zuvor
cars = pd.read_csv("data/cars_multivariate.csv")
cars.head()

## Einfache lineare Regression



Für unsere einfache lineare Regression modellieren wir die Zielvariable (target variable) `mpg` mit dem Feature `horsepower`. 

In [ ]:
# Zusammenhang zwischen horsepower und mpg plotten
cars.plot("horsepower", "mpg", kind="scatter");

Wie gewohnt müssen wir unsere Daten vorbereiten, indem wir Zielvariable und Feature(s) festlegen. Anders als `scikit-learn` fügt `statsmodels` standardmäßig keinen konstanten Term zur Berechnung des Achsenabschnitts (intercept) hinzu. Wenn du dir die Dokumentation von [sklearn.linear_model.LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) ansiehst, findest du dort den Parameter `fit_intercept`, dessen Standardwert `True` ist. Bei [statsmodels OLS](https://www.statsmodels.org/dev/generated/statsmodels.regression.linear_model.OLS.html) müssen wir den konstanten Term selbst hinzufügen, wenn unser Modell mit Achsenabschnitt gefittet werden soll.

In [ ]:
# Zielvariable und Feature festlegen und konstanten Term hinzufügen
X = cars.horsepower
X = sm.add_constant(X)
y = cars.mpg

X.head()

In [ ]:
# Modell fitten und Zusammenfassung ausgeben
model = sm.OLS(y, X)
results = model.fit()

# Achsenabschnitt und Koeffizienten ausgeben
results.params

Wie du siehst, ist die Syntax etwas anders, liefert aber dieselben Ergebnisse. Eine Funktion, die `statsmodels` für statistische Analysen unglaublich nützlich macht, ist `.summary()`, die du für jedes Modell ausgeben kannst. Sie gibt dir einen Überblick über fast alle denkbaren Metriken und statistischen Kennzahlen, die du brauchen könntest. Am Anfang kann das ziemlich überwältigend sein.  

In [ ]:
# Ausführliche Zusammenfassung ausgeben
results.summary()

### Interpretation des Modells



Wir gehen hier nur kurz auf diese Zusammenfassung ein. Eine ausführlichere Erklärung aller Metriken in der Modellzusammenfassung findest du am Ende des Notebooks.

<center><img src="assets/ols_summary.png" height="400"/></center>

* Der blaue Teil der Tabelle enthält einige Angaben zu den Daten und zum Modell selbst.
* Der orange Teil liefert Informationen zur Anpassungsgüte (goodness of fit).
    * **$R^2$**
    * **adjustiertes $R^2$ (adjusted $R^2$)**
* Der grüne Teil enthält Informationen zu den geschätzten Koeffizienten unseres Modells.
    * **coef** sind der Wert für den Achsenabschnitt (intercept), also der Koeffizient der Konstante, sowie die Koeffizienten für jedes Feature
    * **P>|t|** ist eine der wichtigsten Kennzahlen in der Zusammenfassung. Sie nutzt die t-Statistik, um den p-Wert zu berechnen: ein Maß dafür, wie wahrscheinlich es ist, dass dein Koeffizient durch unser Modell nur zufällig so gemessen wird.
* Der türkise Teil unten liefert Informationen zu den Residuen (residuals), zur Autokorrelation (autocorrelation) und zur Multikollinearität (multicollinearity). 

## Multiple lineare Regression



Wir können `statsmodels` auch nutzen, um eine multiple lineare Regression mit mehreren Features zu trainieren. 

In [ ]:
# Zielvariable und Features festlegen und einen konstanten Term hinzufügen
X2 = cars[["horsepower", "weight"]]
X2 = sm.add_constant(X2)
y2 = cars.mpg

X2.head(n=2)

In [ ]:
# Wir können alle Schritte auch per Punktnotation (dot notation) verketten
sm.OLS(y2, X2).fit().summary()

### Vorhersagen für neue Beobachtungen



Wie `scikit-learn` hat auch `statsmodels` eine `.predict`-Methode, um Werte für neue Instanzen zu berechnen. Angenommen, wir wollen die `mpg` für ein neues Auto mit `horsepower` von 250 und einem `weight` von 3000 vorhersagen.
Wir können unsere neuen Daten als Liste übergeben, müssen aber darauf achten, für die Berechnung des Achsenabschnitts eine Konstante von 1 hinzuzufügen. 

In [ ]:
# Vorhersage für das Testauto berechnen
test_car = [1, 250, 3000]
sm.OLS(y2, X2).fit().predict(test_car)

Laut unserem Modell hätte ein Auto mit den oben genannten Eigenschaften also eine Kraftstoffeffizienz von ~15.9 mpg. 

## Die Formel-Notation (formula notation)


Neben der Syntax, die wir oben verwendet haben, können wir auch die sogenannte **Formel-Notation (formula notation)** wählen. Sie liefert dieselben Ergebnisse, nur der Code ist anders und erinnert ein wenig an die Syntax von R.

In [ ]:
# mpg erklärt durch horsepower und weight
smf.ols(formula="mpg ~ horsepower + weight", data=cars).fit().summary()

## Die Modellzusammenfassung, verständlich erklärt...



Hier folgt eine kurze Beschreibung der Zusammenfassungstabelle. Keine Sorge, du musst nicht alles sofort kennen oder verstehen.


#### Der blaue Teil der Tabelle enthält einige Angaben zu den Daten und zum Modell:


* **Dep. Variable**: Abhängige Variable (dependent variable), also die Zielvariable (target)
* **Model**: Verwendete Technik, eine abgekürzte Bezeichnung der Methode
* **Method**: Die Verlustfunktion (loss function), die bei der Wahl der Parameter optimiert wird. Least Squares (Methode der kleinsten Quadrate), weil die Parameter so gewählt werden, dass der Trainingsfehler möglichst klein wird. Das ist auch als mittlerer quadratischer Fehler (Mean Square Error) [MSE] bekannt.
* **No. Observations**: Anzahl der Beobachtungen, mit denen das Modell trainiert wurde, also die Größe der Trainingsdaten
* **Df Residuals**: Freiheitsgrade (degrees of freedom) der Residuen, also Anzahl der Beobachtungen minus Anzahl der Parameter. Der Achsenabschnitt (intercept) zählt als Parameter. Die Freiheitsgrade sollen den Einfluss deskriptiver bzw. zusammenfassender Kennzahlen im Modell abbilden, in der Regression sind das die Koeffizienten. Da die Beobachtungen diesen Parametern "gerecht werden" müssen, bleiben nur entsprechend viele freie Beobachtungen übrig, der Rest ist dafür reserviert, der "Vorgabe" der Parameter "gerecht zu werden". Dieser interne Mechanismus stellt sicher, dass genügend Beobachtungen vorhanden sind, um zu den Parametern zu passen.
* **Df Model**: Die Freiheitsgrade des Modells geben die Anzahl unserer erklärenden Variablen an. Also die Anzahl der Parameter im Modell (ohne den konstanten Term bzw. Achsenabschnitt, falls vorhanden)
* **Covariance Type**: Robuste Regressionsmethoden sind so gebaut, dass sie von Verletzungen der Annahmen über den datenerzeugenden Prozess nicht übermäßig beeinflusst werden. Da dieses Modell Ordinary Least Squares (OLS, gewöhnliche Methode der kleinsten Quadrate) ist, ist es nicht robust und reagiert daher stark auf Ausreißer (outliers).

#### Der orange Teil der Tabelle zeigt die Anpassungsgüte (goodness of fit):


* **R-squared**: Das Bestimmtheitsmaß R² (coefficient of determination). Es gibt an, welcher Prozentsatz der Varianz der Zielvariable durch das Modell erklärt wird. Der verbleibende Prozentsatz ist die Varianz, die auf den Fehler entfällt, den E-Term: der Teil, den Modell und Prädiktoren nicht erfassen.
* **Adj. R-squared**: Das adjustierte R² (adjusted R²), eine Variante des R², die zusätzliche unabhängige Variablen bestraft.
* **F-statistic**: Ein Maß dafür, wie signifikant der Fit ist. Der mittlere quadratische Fehler des Modells geteilt durch den mittleren quadratischen Fehler der Residuen. Fließt in die Berechnung des p-Werts ein.
* **Prob (F-statistic) or P-Value**: Die Wahrscheinlichkeit, dass eine Stichprobe wie diese die obige Statistik liefern würde, und ob das Urteil des Modells über die Nullhypothese (null hypothesis) die Grundgesamtheit (population) verlässlich abbildet. Misst nicht die Größe des Effekts, sondern die Belastbarkeit und Konsistenz dieses Tests auf dieser Datengruppe.
* **Log-likelihood**: Der Logarithmus der Likelihood-Funktion. Er misst, wie wahrscheinlich es ist, dass die beobachteten Daten unter dem Modell auftreten. Höhere Log-Likelihood-Werte zeigen, dass das Modell besser zu den Daten passt.
* **AIC**: Das Akaike-Informationskriterium (Akaike Information Criterion). Passt die Log-Likelihood anhand der Anzahl der Beobachtungen und der Komplexität des Modells an. Bestraft die Kennzahlen zur Modellauswahl, wenn weitere unabhängige Variablen hinzukommen.
* **BIC**: Das Bayessche Informationskriterium (Bayesian Information Criterion). Ähnlich wie das AIC, bestraft Modelle mit mehr Parametern aber stärker. Bestraft die Kennzahlen zur Modellauswahl, wenn weitere unabhängige Variablen hinzukommen.


#### Die grüne zweite Tabelle zeigt den Koeffizienten-Bericht:


* **coef**: Der geschätzte Wert des Koeffizienten. Also der Faktor, mit dem das Modell den Wert der unabhängigen Variable multipliziert.
std err: Der einfache Standardfehler (standard error) der Schätzung des Koeffizienten. Die durchschnittliche Abweichung der Punkte vom Modell. Das bietet eine Möglichkeit, die Modellgenauigkeit in der passenden Einheit einzuschätzen.
* **t**: Der Wert der t-Statistik. Er misst, wie statistisch signifikant der Koeffizient ist.
* **P > |t|**: p-Wert dafür, dass die Nullhypothese (null hypothesis) "der Koeffizient = 0" wahr ist. Ist er kleiner als das Konfidenzniveau, oft 0.05, deutet das auf einen statistisch signifikanten Zusammenhang zwischen dem Term und der Zielgröße (response) hin.
* **[95.0% Conf. Interval]**: Der untere und obere Wert des 95%-Konfidenzintervalls (confidence interval). Der konkrete Bereich möglicher Werte des Koeffizienten.


#### Die türkise dritte Tabelle liefert Informationen zu Residuen, Autokorrelation und Multikollinearität:












* **Skewness**: Die Schiefe. Ein Maß für die Symmetrie der Daten um den Mittelwert. Normalverteilte Fehler sollten symmetrisch um den Mittelwert verteilt sein (gleich viele oberhalb und unterhalb der Linie). Die Normalverteilung hat eine Schiefe von 0.
* **Kurtosis**: Die Wölbung. Ein Maß für die Form der Verteilung. Vergleicht die Menge der Daten nahe am Mittelwert mit der Menge weit entfernt vom Mittelwert (in den Rändern, tails), also wie "spitz" die Verteilung ist. Die Normalverteilung hat eine Kurtosis von 3, und je größer der Wert, desto spitzer ist die Kurve.
* **Omnibus D’Angostino’s test**: Ein kombinierter statistischer Test auf das Vorliegen von Schiefe und Wölbung.
* **Prob(Omnibus)**: Die obige Statistik, umgerechnet in eine Wahrscheinlichkeit
* **Jarque-Bera**: Ein anderer Test auf Schiefe und Wölbung
* **Prob (JB)**: Die obige Statistik, umgerechnet in eine Wahrscheinlichkeit
* **Durbin-Watson**: Ein Test auf Autokorrelation (also darauf, dass die Fehler nicht unabhängig sind), was oft in der Zeitreihenanalyse wichtig ist. Er ist auch ein Maß für Homoskedastizität (homoscedasticity). Ein idealer Wert liegt zwischen 1 und 2.
* **Cond. No**: Konditionszahl (condition number). Ein Test auf Multikollinearität (also ob bei einem Fit mit mehreren Parametern die Parameter miteinander zusammenhängen). Eine hohe Konditionszahl deutet stark auf Multikollinearität hin.